In [130]:
import pandas as pd
import json
import pickle
from tqdm.notebook import tqdm

# Decompress ClassyFire

In [ ]:
import zstandard as zstd

input_path = 'data/classyfire.v4.tsv.zst' # -> 2.1 GB
output_path = 'data/classyfire.v4.tsv'    # -> ~24 GB

file_size = __import__('os').path.getsize(input_path)
    
decomp = zstd.ZstdDecompressor()

with open(input_path, 'rb') as f_in, \
        open(output_path, 'wb') as f_out, \
        decomp.stream_reader(f_in) as reader, \
        tqdm(total=file_size, unit='B', unit_scale=True, desc="Decompressing") as pbar:
    
    while True:
        chunk = reader.read(1024*1024)
        while chunk:
            f_out.write(chunk)
            pbar.update(len(chunk))

print("Done!")

# MassSpecGym dataset

## Overview

In [6]:
msg_df = pd.read_csv('data/MassSpecGym1.5.tsv', sep='\t')
print(msg_df.shape)
msg_df.info()

(231104, 14)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 231104 entries, 0 to 231103
Data columns (total 14 columns):
 #   Column                Non-Null Count   Dtype  
---  ------                --------------   -----  
 0   identifier            231104 non-null  object 
 1   mzs                   231104 non-null  object 
 2   intensities           231104 non-null  object 
 3   smiles                231104 non-null  object 
 4   inchikey              231104 non-null  object 
 5   formula               231104 non-null  object 
 6   precursor_formula     231104 non-null  object 
 7   parent_mass           231104 non-null  float64
 8   precursor_mz          231104 non-null  float64
 9   adduct                231104 non-null  object 
 10  instrument_type       225881 non-null  object 
 11  collision_energy      121746 non-null  float64
 12  fold                  231104 non-null  object 
 13  simulation_challenge  231104 non-null  bool   
dtypes: bool(1), float64(3), object(10)
memo

In [7]:
msg_df['fold'].value_counts()

fold
train    194119
val       19429
test      17556
Name: count, dtype: int64

## Processing

In [55]:
test_msg_df = msg_df[msg_df['fold'] == 'test']
print(test_msg_df.shape)
test_msg_df.info()

(17556, 14)
<class 'pandas.core.frame.DataFrame'>
Index: 17556 entries, 168 to 231100
Data columns (total 14 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   identifier            17556 non-null  object 
 1   mzs                   17556 non-null  object 
 2   intensities           17556 non-null  object 
 3   smiles                17556 non-null  object 
 4   inchikey              17556 non-null  object 
 5   formula               17556 non-null  object 
 6   precursor_formula     17556 non-null  object 
 7   parent_mass           17556 non-null  float64
 8   precursor_mz          17556 non-null  float64
 9   adduct                17556 non-null  object 
 10  instrument_type       17147 non-null  object 
 11  collision_energy      10162 non-null  float64
 12  fold                  17556 non-null  object 
 13  simulation_challenge  17556 non-null  bool   
dtypes: bool(1), float64(3), object(10)
memory usage: 1.9+ MB


In [ ]:
test_msg_df['inchikey'].value_counts()

inchikey
RWXIFXNRCLMQCD    383
QOVKKEKBURQILF    194
NWXMGUDVXFXRIG    159
RXZBMPWDPOLZGW    139
ZVTVWDXRNMHGNY    126
                 ... 
LBEJUQWGEGOTNX      1
QGVLYPPODPLXMB      1
WEGITEKALFVLHV      1
KVHRYLNQDWXAGI      1
HVKUYPXKTAMIFI      1
Name: count, Length: 2998, dtype: int64

In [56]:
dedupl_msg_df = test_msg_df.drop_duplicates('inchikey')
print(dedupl_msg_df.shape)
dedupl_msg_df.info()

(2998, 14)
<class 'pandas.core.frame.DataFrame'>
Index: 2998 entries, 168 to 222364
Data columns (total 14 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   identifier            2998 non-null   object 
 1   mzs                   2998 non-null   object 
 2   intensities           2998 non-null   object 
 3   smiles                2998 non-null   object 
 4   inchikey              2998 non-null   object 
 5   formula               2998 non-null   object 
 6   precursor_formula     2998 non-null   object 
 7   parent_mass           2998 non-null   float64
 8   precursor_mz          2998 non-null   float64
 9   adduct                2998 non-null   object 
 10  instrument_type       2753 non-null   object 
 11  collision_energy      2269 non-null   float64
 12  fold                  2998 non-null   object 
 13  simulation_challenge  2998 non-null   bool   
dtypes: bool(1), float64(3), object(10)
memory usage: 330.8+ KB


# Enrich MassSepcGym dataset with ClassyFire ChemONT trees

## Create pickle with ChemONT trees of interest

In [ ]:
chemont_trees_cache_path = "data/chemont_cache.pkl"

In [177]:
inchikeys = list(test_msg_df['inchikey'].drop_duplicates().values)
aux_d = {
    'inchikey': dict(),
    'chemont_tree': dict()
    }

for i, k in enumerate(inchikeys):
    aux_d['inchikey'][i] = k
    aux_d['chemont_tree'][i] = None

key_chemont_df = pd.DataFrame.from_dict(aux_d)

In [ ]:
classyfire_dfs = pd.read_csv('data/classyfire.v4.tsv', sep='\t', chunksize=20000)

for key_chemont_df in tqdm(classyfire_dfs):
    key_chemont_df['inchikey'] = key_chemont_df['inchikey'].apply(lambda key: key.split("-")[0])  # Keep only first block of the Inchikey
    key_chemont_df.drop_duplicates('inchikey', inplace=True)
    key_chemont_df = key_chemont_df[['inchikey', 'chemont_tree_json']]

    key_chemont_df = pd.merge(key_chemont_df, key_chemont_df, on='inchikey', how='left')

    key_chemont_df['chemont_tree'].where(key_chemont_df['chemont_tree'].notnull(), key_chemont_df['chemont_tree_json'], inplace=True)

    key_chemont_df.drop('chemont_tree_json', axis=1, inplace=True)

with open(chemont_trees_cache_path, "wb") as temp:
    key_chemont_df.to_pickle(temp)

key_chemont_df.info()
key_chemont_df

0it [00:00, ?it/s]

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2998 entries, 0 to 2997
Data columns (total 2 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   inchikey      2998 non-null   object
 1   chemont_tree  2987 non-null   object
dtypes: object(2)
memory usage: 47.0+ KB


,inchikey,chemont_tree
0,GYSCAQFHASJXRS,"[0,264,1813,1961,1994]"
1,OUMWCYMRLMEZJH,"[1,423,453,554,554]"
2,ZRTGPZGAMCJZNA,"[0,12,259,1550,1550]"
3,HWSQVPGTQUYLEQ,"[0,12,3909,1334,2518]"
4,MXLWQNCWIIZUQT,"[0,2448,2903,2903,2903]"
...,...,...
2993,XMSXQFUHVRWGNA,"[0,462,1370,3194,4445]"
2994,HXIQYSLFEXIOAV,"[1,423,453,555,555]"
2995,LYUPEIXJYAJCHL,"[0,12,259,1551,3788]"
2996,PMCAUYATZKXGHU,"[0,12,259,1551,3788]"


In [ ]:
with open(chemont_trees_cache_path, "rb") as f:
    key_chemont_df = pickle.load(f)

key_chemont_df[key_chemont_df['chemont_tree'].isnull()]

,inchikey,chemont_tree
2837,PCAYKUNUODABRQ,NaN
2905,YNMMSSYXBYTABV,NaN
2906,HPZNCFSLZGFDST,NaN
2907,ATASRQYZQYFECN,NaN
2908,LVFIUDAMNWXFMK,NaN
2909,HDNFKWOIOAMTET,NaN
2916,LGSJLBWWZAOHIV,NaN
2923,NXPWZDQBJBVYGK,NaN
2927,IQZKGDLOGOUDKG,NaN
2928,GOMIKTMVKFBMCU,NaN


## Join MSG dataset with ChemONT trees

In [185]:
target_msg_df = test_msg_df.copy()

result_df = pd.merge(target_msg_df, key_chemont_df, on='inchikey')

result_df.info()
result_df[result_df['chemont_tree'].isnull()]


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 17556 entries, 0 to 17555
Data columns (total 15 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   identifier            17556 non-null  object 
 1   mzs                   17556 non-null  object 
 2   intensities           17556 non-null  object 
 3   smiles                17556 non-null  object 
 4   inchikey              17556 non-null  object 
 5   formula               17556 non-null  object 
 6   precursor_formula     17556 non-null  object 
 7   parent_mass           17556 non-null  float64
 8   precursor_mz          17556 non-null  float64
 9   adduct                17556 non-null  object 
 10  instrument_type       17147 non-null  object 
 11  collision_energy      10162 non-null  float64
 12  fold                  17556 non-null  object 
 13  simulation_challenge  17556 non-null  bool   
 14  chemont_tree          17544 non-null  object 
dtypes: bool(1), float64

,identifier,mzs,intensities,smiles,inchikey,formula,precursor_formula,parent_mass,precursor_mz,adduct,instrument_type,collision_energy,fold,simulation_challenge,chemont_tree
17353,MassSpecGymID0396217,"71.049797,83.049644,83.084198,95.047134,97.063...","0.011825309995339752,0.011819240859169352,0.00...",CCCCC/C=C/C=C\C(=O)O[C@@H]1[C@@H](C)[C@]2(O)[C...,PCAYKUNUODABRQ,C34H46O7,C34H46NaO7,566.323782,589.313,[M+Na]+,QTOF,NaN,test,False,NaN
17354,MassSpecGymID0396218,"71.049797,83.049644,83.084198,95.047134,97.063...","0.011825309995339752,0.011819240859169352,0.00...",CCCCC/C=C/C=C\C(=O)O[C@@H]1[C@@H](C)[C@]2(O)[C...,PCAYKUNUODABRQ,C34H46O7,C34H46NaO7,566.327782,589.317,[M+Na]+,QTOF,NaN,test,False,NaN
17450,MassSpecGymID0397635,"52.847301,68.049599,77.991096,80.049797,95.012...","0.004090909090909091,0.005681818181818182,0.00...",CC(=O)OC1C(OC(=O)c2ccc(=O)n(C)c2)CC(C)C23OC(C)...,YNMMSSYXBYTABV,C37H44N2O14,C37H45N2O14,740.279724,741.287,[M+H]+,Orbitrap,NaN,test,False,NaN
17451,MassSpecGymID0397636,"57.070499,59.049702,70.788002,80.049698,85.064...","0.066,0.0037,0.0037,0.011,0.0074,0.004,0.02,0....",CC[C@@H](C)C(=O)OC[C@]12C(=O)[C@@H](OC(=O)c3cc...,HPZNCFSLZGFDST,C36H44N2O13,C36H45N2O13,712.224724,713.232,[M+H]+,Orbitrap,NaN,test,False,NaN
17452,MassSpecGymID0397637,"57.070499,80.049599,95.013,108.0448,136.039505...","0.024390243902439025,0.008048780487804878,0.01...",CC[C@@H](C)C(=O)OC[C@]12C(=O)C(OC(=O)c3ccc(=O)...,ATASRQYZQYFECN,C38H46N2O14,C38H47N2O14,754.294724,755.302,[M+H]+,Orbitrap,NaN,test,False,NaN
17453,MassSpecGymID0397638,"57.070599,59.049801,85.065102,95.012802,105.07...","0.16756756756756758,0.02,0.023513513513513513,...",CC[C@@H](C)C(=O)OC[C@]12[C@H](OC(=O)[C@@H](C)C...,LVFIUDAMNWXFMK,C41H54N2O14,C41H55N2O14,798.357724,799.365,[M+H]+,Orbitrap,NaN,test,False,NaN
17454,MassSpecGymID0397639,"59.049801,68.050201,71.049698,80.050102,83.049...","0.00411764705882353,0.004294117647058823,0.011...",CC[C@H](C)C(=O)O[C@@H]1[C@H](OC(=O)c2ccc(=O)n(...,HDNFKWOIOAMTET,C42H54N2O15,C42H55N2O15,826.352724,827.360,[M+H]+,Orbitrap,NaN,test,False,NaN
17461,MassSpecGymID0397815,"55.019798,76.023003,84.044502,130.050201,131.0...","0.04905660377358491,0.09056603773584905,0.1660...",N[C@@H](CCC(=O)N[C@H]1CSSC[C@@H](C(=O)NCC(=O)O...,LGSJLBWWZAOHIV,C18H27N5O10S2,C18H28N5O10S2,537.122724,538.130,[M+H]+,QTOF,NaN,test,False,NaN
17468,MassSpecGymID0397912,"51.695801,54.804001,54.806301,56.807701,56.918...","0.0033333333333333335,0.0021538461538461538,0....",CCC1=C(C)C2=NC1=CC1=C(C)C3=C(O)[C@H](O)C(=C4NC...,NXPWZDQBJBVYGK,C32H34N4O5,C32H35N4O5,554.252724,555.260,[M+H]+,Orbitrap,NaN,test,False,NaN
17473,MassSpecGymID0397917,"50.022499,51.577702,51.5798,51.6917,53.390598,...","0.0013225806451612903,0.0013548387096774194,0....",CCC1=C(C)C2=NC1=CC1=C(C)C3=C(O)[C@H](C(=O)OC)C...,IQZKGDLOGOUDKG,C34H36N4O6,C34H37N4O6,596.262724,597.270,[M+H]+,Orbitrap,NaN,test,False,NaN
